# Investment Property Financial Model

This notebook builds a 10-year financial model for a property investment in Saratoga,
computing rental cash flows, debt service, IRR, and sale proceeds.

In [ ]:
import os

# ===== Read introduction to extract parameters =====
data_paths = ['/workspace/data/', './environment/data/', '../environment/data/',
              '/home/ddzhang/DSBench/colab_tasks/dsbench_da_00000041/environment/data/']

data_dir = None
for p in data_paths:
    if os.path.exists(p):
        data_dir = p
        break

if data_dir:
    with open(os.path.join(data_dir, 'introduction.txt'), 'r') as f:
        intro_text = f.read()
    print('Introduction loaded from:', data_dir)
    print(intro_text[:300])
else:
    print('Data directory not found; using parameters from problem statement.')
    intro_text = ''

# Read questions
if data_dir:
    for i in range(1, 6):
        qpath = os.path.join(data_dir, f'question{i}.txt')
        if os.path.exists(qpath):
            with open(qpath, 'r') as f:
                print(f'\nQuestion {i}:', f.read().strip())

In [ ]:
# ===== Model Parameters (extracted from introduction.txt) =====

purchase_price = 1_600_000         # Apartment purchase price
furniture_cost = 10_000            # Budget to furnish apartment
mortgage_fee = 10_000              # Bank arrangement fee

# Total cost basis for Capital Gains Tax calculation
total_cost_basis = purchase_price + mortgage_fee + furniture_cost  # $1,620,000

max_loan = 900_000                 # Maximum mortgage loan
mortgage_rate = 0.05               # Fixed interest rate (5%)
mortgage_term_years = 25           # Mortgage term

initial_rent = 83_200              # Starting annual rental income
agency_fee_rate = 0.09             # Letting agent fees (% of rent)
maintenance_rate = 0.05            # Maintenance costs (% of rent)
income_tax_rate = 0.35             # Income tax rate
cgt_rate = 0.20                    # Capital gains tax rate

# Property doubles in 10 years with constant compound growth
# purchase_price * (1+g)^10 = 2 * purchase_price => g = 2^(1/10) - 1
growth_rate = 2 ** (1.0 / 10) - 1

# Equity invested at Year 0 (purchase price + fees - mortgage)
equity_invested = purchase_price + furniture_cost + mortgage_fee - max_loan

print(f'Purchase price:          ${purchase_price:>12,}')
print(f'Furniture cost:          ${furniture_cost:>12,}')
print(f'Mortgage arrangement fee:${mortgage_fee:>12,}')
print(f'Total cost basis (CGT):  ${total_cost_basis:>12,}')
print(f'Loan amount:             ${max_loan:>12,}')
print(f'Equity invested (Year 0):${equity_invested:>12,}')
print(f'Property growth rate:    {growth_rate*100:.4f}% per annum')
print(f'Initial annual rent:     ${initial_rent:>12,}')

In [ ]:
# ===== Build Year-by-Year Financial Model =====
#
# Rental income growth: 2% for the first two growth periods (years 2 and 3),
# then 5% growth from year 4 onwards.
#
# Cash flow logic:
# 1. Rental income is received
# 2. Agency fees (9% of rent) and maintenance (5% of rent) are paid
# 3. Interest is charged on opening debt balance
# 4. Tax = (rent - agency - maintenance - interest) * 35%
# 5. Cash available = rent - agency - maintenance - tax
# 6. After paying interest, ALL remaining cash goes to principal repayment

results = []
debt_opening = max_loan

for year in range(1, 11):
    # --- Rental Income ---
    # Year 1: base rent $83,200
    # Year 2: +2% growth
    # Year 3: +2% growth (second year of 2% increases)
    # Year 4+: +5% growth each year
    if year == 1:
        rent = initial_rent
    elif year <= 3:
        # 2% growth for years 2 and 3
        rent = initial_rent * (1.02 ** (year - 1))
    else:
        # After two years of 2%, switch to 5% growth
        # Year 3 rent = initial * 1.02^2
        # Year 4 = Year3 * 1.05, Year 5 = Year4 * 1.05, etc.
        rent = initial_rent * (1.02 ** 2) * (1.05 ** (year - 3))

    # --- Operating Expenses ---
    agency_fees = rent * agency_fee_rate
    maintenance = rent * maintenance_rate

    # --- Interest on outstanding debt ---
    interest = debt_opening * mortgage_rate

    # --- Tax (interest is tax deductible) ---
    taxable_income = rent - agency_fees - maintenance - interest
    tax = taxable_income * income_tax_rate

    # --- Cash Flow Available for Debt Service ---
    # This is the net cash after operating expenses and tax, before debt payments
    net_cash = rent - agency_fees - maintenance - tax

    # Cash flow available for debt service (after interest = principal repayment capacity)
    cfads = net_cash - interest

    # --- Debt Service ---
    # All free cash after interest goes to principal repayment
    principal_repayment = max(0, min(cfads, debt_opening))
    debt_closing = debt_opening - principal_repayment

    # --- Property Value ---
    property_value = purchase_price * (1 + growth_rate) ** year

    # --- Sale Analysis ---
    capital_gain = property_value - total_cost_basis
    cgt = capital_gain * cgt_rate if capital_gain > 0 else 0
    sale_proceeds = property_value - cgt - debt_closing

    results.append({
        'year': year,
        'rent': rent,
        'agency_fees': agency_fees,
        'maintenance': maintenance,
        'interest': interest,
        'taxable_income': taxable_income,
        'tax': tax,
        'net_cash': net_cash,
        'cfads': cfads,
        'principal_repayment': principal_repayment,
        'debt_opening': debt_opening,
        'debt_closing': debt_closing,
        'property_value': property_value,
        'capital_gain': capital_gain,
        'cgt': cgt,
        'sale_proceeds': sale_proceeds,
    })

    debt_opening = debt_closing

# Print summary table
print(f"{'Year':>4} {'Rent':>12} {'CFADS':>10} {'Interest':>10} {'Princ Rep':>10} {'Debt Close':>12} {'Prop Value':>14} {'Sale Proceeds':>14}")
print('-' * 100)
for r in results:
    print(f"{r['year']:>4} {r['rent']:>12,.2f} {r['cfads']:>10,.2f} {r['interest']:>10,.2f} "
          f"{r['principal_repayment']:>10,.2f} {r['debt_closing']:>12,.2f} "
          f"{r['property_value']:>14,.2f} {r['sale_proceeds']:>14,.2f}")

In [ ]:
# ===== Question 1 =====
# What is the Cash Flow Available for Debt Service at end of first year?
# Options: a. 17,259  b. 18,992  c. 19,211  d. 19,396

cfads_year1 = results[0]['cfads']
print(f'Cash Flow Available for Debt Service (Year 1): ${cfads_year1:,.2f}')

# Match to closest option
q1_options = {'A': 17259, 'B': 18992, 'C': 19211, 'D': 19396}
q1_answer = min(q1_options, key=lambda k: abs(q1_options[k] - cfads_year1))
print(f'Closest option: {q1_answer} (${q1_options[q1_answer]:,})')
print(f'Difference: ${abs(q1_options[q1_answer] - cfads_year1):,.2f}')

In [ ]:
# ===== Question 2 =====
# What is the estimated closing debt principal outstanding at end of year 4?
# Options: a. 820,296  b. 822,122  c. 824,825  d. 826,020

debt_close_year4 = results[3]['debt_closing']
print(f'Closing debt (Year 4): ${debt_close_year4:,.2f}')

q2_options = {'A': 820296, 'B': 822122, 'C': 824825, 'D': 826020}
q2_answer = min(q2_options, key=lambda k: abs(q2_options[k] - debt_close_year4))
print(f'Closest option: {q2_answer} (${q2_options[q2_answer]:,})')
print(f'Difference: ${abs(q2_options[q2_answer] - debt_close_year4):,.2f}')

In [ ]:
# ===== IRR Calculation Utility =====

def compute_irr(cashflows, tol=1e-12, max_iter=5000):
    """Compute Internal Rate of Return using bisection method."""
    def npv(rate):
        return sum(cf / (1 + rate) ** t for t, cf in enumerate(cashflows))

    lo, hi = -0.5, 10.0
    if npv(lo) < 0:
        return None  # No valid IRR in range
    for _ in range(max_iter):
        mid = (lo + hi) / 2
        if npv(mid) > 0:
            lo = mid
        else:
            hi = mid
        if hi - lo < tol:
            break
    return (lo + hi) / 2

print('IRR utility defined.')

In [ ]:
# ===== Question 3 =====
# What is the expected IRR if property is sold at end of 10 years?
# Options: a. 9.2%  b. 10.2%  c. 11.2%  d. 12.2%
#
# Cash flows from investor's perspective:
#   Year 0: -equity_invested (cash outflow to buy property)
#   Years 1-9: $0 (all free cash used to pay down mortgage)
#   Year 10: sale_proceeds (property value - CGT - remaining debt)

cf_10 = [-equity_invested] + [0] * 9 + [results[9]['sale_proceeds']]
irr_10 = compute_irr(cf_10)
print(f'Sale proceeds at Year 10: ${results[9]["sale_proceeds"]:,.2f}')
print(f'IRR (sell at Year 10): {irr_10*100:.2f}%')

q3_options = {'A': 9.2, 'B': 10.2, 'C': 11.2, 'D': 12.2}
q3_answer = min(q3_options, key=lambda k: abs(q3_options[k] - irr_10 * 100))
print(f'Closest option: {q3_answer} ({q3_options[q3_answer]}%)')

In [ ]:
# ===== Question 4 =====
# If sold after 2 years, expected proceeds after satisfying CGT and repaying debt?
# Options: a. $930,341  b. $930,343  c. $930,345  d. $930,347

sale_proceeds_year2 = results[1]['sale_proceeds']
print(f'Property value at Year 2: ${results[1]["property_value"]:,.2f}')
print(f'Capital gains tax:        ${results[1]["cgt"]:,.2f}')
print(f'Closing debt:             ${results[1]["debt_closing"]:,.2f}')
print(f'Sale proceeds (Year 2):   ${sale_proceeds_year2:,.2f}')

q4_options = {'A': 930341, 'B': 930343, 'C': 930345, 'D': 930347}
q4_answer = min(q4_options, key=lambda k: abs(q4_options[k] - sale_proceeds_year2))
print(f'Closest option: {q4_answer} (${q4_options[q4_answer]:,})')
print(f'Difference: ${abs(q4_options[q4_answer] - sale_proceeds_year2):,.2f}')

In [ ]:
# ===== Question 5 =====
# The highest IRR achieved is by exiting in which year?
# Options: a. Year 2  b. Year 5  c. Year 6  d. Year 8
#
# For each possible exit year, compute IRR:
#   Year 0: -equity_invested
#   Years 1 to (year-1): 0
#   Year N: sale_proceeds at that year

irrs = {}
for year_idx in range(10):
    year = year_idx + 1
    cf = [-equity_invested] + [0] * (year - 1) + [results[year_idx]['sale_proceeds']]
    irr_val = compute_irr(cf)
    irrs[year] = irr_val
    print(f'  Year {year}: IRR = {irr_val*100:.2f}%, Sale proceeds = ${results[year_idx]["sale_proceeds"]:,.2f}')

best_year = max(irrs, key=irrs.get)
print(f'\nBest IRR achieved in Year {best_year}: {irrs[best_year]*100:.2f}%')

q5_options = {'A': 2, 'B': 5, 'C': 6, 'D': 8}
q5_answer = min(q5_options, key=lambda k: abs(q5_options[k] - best_year))
print(f'Closest option: {q5_answer} (Year {q5_options[q5_answer]})')

In [ ]:
# ===== Final Answers =====
answers = {
    'question1': q1_answer,
    'question2': q2_answer,
    'question3': q3_answer,
    'question4': q4_answer,
    'question5': q5_answer,
}

print('\n===== FINAL ANSWERS =====')
for k, v in answers.items():
    print(f'  {k}: {v}')

print('\nExpected: Q1=A, Q2=A, Q3=D, Q4=B, Q5=A')